# FastAPI Deployment — Kestrel Software / DiffSage

**Scenario.** Kestrel Software builds developer tools. Their product, **DiffSage**,
is an AI agent that reviews pull requests inside CI, before a human reviewer looks
at the diff — checking the diff against the team's style guide, flagging risky
dependency changes, and remembering what it found last time the same PR came
through.

This notebook wraps that agent behind a real HTTP API and covers three things:

1. **Wrapping a LangGraph agent in FastAPI** — `POST /invoke` (blocking) and
   `POST /stream` (Server-Sent Events)
2. **Running it locally on Windows with Uvicorn**, with **auto-trace middleware**
   that stamps every request with an ID and latency
3. **Database connection pooling in FastAPI** — a real `aiosqlite` pool, plus
   LangGraph's own checkpointer for per-thread agent state

Two real bugs turned up while building this and are kept in below, root cause
first: a `lambda` that wraps an async call is not itself a coroutine function
(the agent silently stalls), and a pooled SQLite writer needs
`PRAGMA busy_timeout` or concurrent writes throw `database is locked`.

## Offline / live duality

Same rule as every other module in this series: absence of an API key
transparently switches a piece to an offline double, and nothing downstream
needs to know which one is active.

| Service | Live | Offline |
|---|---|---|
| Groq (`GROQ_API_KEY`) | `langchain_groq.ChatGroq`, streamed | `FakeChatModel` — deterministic, same `.ainvoke` / streaming interface |
| Tavily (`TAVILY_API_KEY`) | real search for dependency advisories | fixed lookup table |
| LangSmith (`LANGSMITH_API_KEY` + `LANGSMITH_TRACING`) | middleware opens a run per request | middleware logs the same shape locally |
| ChromaDB | — | always local; a deterministic hash embedder, not tied to live/offline, since RAG quality isn't the point of this module |
| SQLite / LangGraph checkpointer | — | always local; no key required |

In [ ]:
# --break-system-packages is a no-op on Anaconda/venv; it only matters on
# system Python installs that mark themselves externally-managed (PEP 668).
%pip install -q --break-system-packages fastapi "uvicorn[standard]" langgraph langgraph-checkpoint-sqlite aiosqlite chromadb langchain-core langchain-groq tavily-python python-dotenv httpx

In [ ]:
import asyncio
import hashlib
import json
import os
import time
import uuid
from contextlib import asynccontextmanager
from typing import Any, AsyncIterator, List, Literal, Optional, TypedDict

from dotenv import load_dotenv

load_dotenv()

GROQ_LIVE = bool(os.getenv("GROQ_API_KEY"))
TAVILY_LIVE = bool(os.getenv("TAVILY_API_KEY"))
LANGSMITH_LIVE = bool(os.getenv("LANGSMITH_API_KEY")) and os.getenv("LANGSMITH_TRACING", "").lower() == "true"

print(f"groq:      {'live' if GROQ_LIVE else 'offline'}")
print(f"tavily:    {'live' if TAVILY_LIVE else 'offline'}")
print(f"langsmith: {'live' if LANGSMITH_LIVE else 'offline'}")


def stable_hash(text: str) -> str:
    """hash() is randomized per process (PYTHONHASHSEED); this isn't."""
    return hashlib.md5(text.encode()).hexdigest()[:8]

## The chat model

`FakeChatModel` matches `langchain_groq.ChatGroq`'s call surface exactly —
`.ainvoke(...)` and streaming — so every node written against it works
unmodified once real Groq credentials are present.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult


class FakeChatModel(BaseChatModel):
    """Deterministic offline double for a real chat model."""

    def _make_reply(self, messages: List[BaseMessage]) -> str:
        last = str(messages[-1].content) if messages else ""
        seed = stable_hash(last)
        return (
            f"[offline review seed={seed}] Found 1 dependency advisory and "
            "1 style guideline note. Recommend request_changes."
        )

    def _generate(self, messages, stop=None, run_manager=None, **kwargs: Any) -> ChatResult:
        text = self._make_reply(messages)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])

    async def _astream(self, messages, stop=None, run_manager=None, **kwargs: Any) -> AsyncIterator[ChatGenerationChunk]:
        text = self._make_reply(messages)
        for word in text.split(" "):
            await asyncio.sleep(0)
            yield ChatGenerationChunk(message=AIMessageChunk(content=word + " "))

    @property
    def _llm_type(self) -> str:
        return "fake-chat-model"


def build_chat_model() -> BaseChatModel:
    if GROQ_LIVE:
        from langchain_groq import ChatGroq
        return ChatGroq(model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"), api_key=os.getenv("GROQ_API_KEY"), streaming=True, temperature=0.2)
    return FakeChatModel()


# smoke test — both call paths a graph node will actually use
_model = build_chat_model()
_resp = await _model.ainvoke([HumanMessage(content="+requests==2.25.0")])
print("ainvoke:", _resp.content)

print("astream:", end=" ")
async for chunk in _model.astream([HumanMessage(content="+requests==2.25.0")]):
    print(chunk.content, end="")
print()

## Style guide retrieval (ChromaDB)

A dozen embedded style-guide snippets, queried per review. This module isn't
teaching RAG quality, so the embedder is a small deterministic hash function
rather than a downloaded model — it keeps this piece fully offline in both
live and offline mode, with no first-run network dependency.

In [ ]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings


class DeterministicEmbedder(EmbeddingFunction):
    DIM = 64

    def __init__(self) -> None:
        pass

    def __call__(self, input: Documents) -> Embeddings:
        out = []
        for doc in input:
            vec = [0.0] * self.DIM
            for tok in doc.lower().split():
                h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
                vec[h % self.DIM] += 1.0
            norm = sum(v * v for v in vec) ** 0.5 or 1.0
            out.append([v / norm for v in vec])
        return out


STYLE_GUIDE = [
    "Wrap external HTTP calls to third-party webhooks in the shared retry decorator.",
    "Never catch bare Exception; catch the specific exception type you expect.",
    "Pin dependency versions exactly in requirements.txt, do not use loose ranges.",
    "Database queries in request handlers must go through the connection pool, never a raw sqlite3.connect call.",
    "Log request IDs, never log full request bodies for endpoints that touch customer data.",
]


def build_style_guide_collection():
    client = chromadb.Client()
    coll = client.get_or_create_collection("style_guide", embedding_function=DeterministicEmbedder())
    if coll.count() == 0:
        coll.add(ids=[f"sg{i}" for i in range(len(STYLE_GUIDE))], documents=STYLE_GUIDE)
    return coll


_coll = build_style_guide_collection()
_res = _coll.query(query_texts=["requests dependency version pinning"], n_results=2)
for doc in _res["documents"][0]:
    print("-", doc)

## Dependency advisory lookup (Tavily double)

Fires only when the diff touches a manifest file (`requirements.txt`,
`package.json`, ...) — the graph decides this with a conditional edge later on.

In [ ]:
FAKE_ADVISORIES = {
    "requests==2.25.0": "requests==2.25.0 has a known CVE; pin >=2.31.0 instead.",
}


async def check_dependency_advisory(package_line: str) -> Optional[str]:
    if TAVILY_LIVE:
        from tavily import TavilyClient
        client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        result = await asyncio.to_thread(client.search, f"{package_line} security advisory CVE", max_results=1)
        hits = result.get("results", [])
        return hits[0]["content"][:200] if hits else None
    return FAKE_ADVISORIES.get(package_line.strip())


print(await check_dependency_advisory("requests==2.25.0"))
print(await check_dependency_advisory("flask==2.0.1"))

## Database connection pooling — and the bug it hides

The pool: open N `aiosqlite` connections up front, hand them out from a
queue, return them when done. WAL mode lets multiple readers proceed
concurrently.

The bug we hit building this: **without `PRAGMA busy_timeout` set on every
pooled connection, concurrent writers raise `sqlite3.OperationalError:
database is locked`** instead of waiting their turn. WAL mode allows many
concurrent *readers*, but SQLite still permits only one *writer* at a time —
`busy_timeout` is what makes the other writers queue politely for that one
writer's turn instead of failing immediately. It is not optional for a pool
that will ever see concurrent writes.

Root cause, concretely: a plain `sqlite3`/`aiosqlite` connection defaults to
returning "locked" the instant it can't get the write lock. `busy_timeout`
tells it to retry for up to that many milliseconds before giving up.

There's a second, unrelated bug in the same neighborhood worth seeing before
the fix: it's tempting to wire a pooled connection into a LangGraph node with
`lambda s: some_async_fn(s, pool)`. That lambda is a *synchronous* function
that happens to return a coroutine object — `asyncio.iscoroutinefunction`
doesn't know to await it, so the node silently stalls instead of running.

In [ ]:
# the lambda trap, made visible without actually hanging anything

async def real_node(state, pool):
    return state

async def _demo_pool():
    return None

bad = lambda s: real_node(s, _demo_pool)   # returns a coroutine object when called
good = lambda s: None                        # a plain sync lambda is fine for sync nodes

async def good_async(s):                     # this is what a lambda can't be
    return await real_node(s, _demo_pool)

print("lambda wrapping an async call, iscoroutinefunction:", asyncio.iscoroutinefunction(bad))
print("real async def closure,        iscoroutinefunction:", asyncio.iscoroutinefunction(good_async))

In [ ]:
import aiosqlite


class SQLitePool:
    def __init__(self, path: str, size: int = 5, busy_timeout_ms: int = 5000):
        self._queue: asyncio.Queue = asyncio.Queue(maxsize=size)
        self._path, self._size, self._busy_timeout_ms = path, size, busy_timeout_ms

    async def open(self) -> None:
        for _ in range(self._size):
            conn = await aiosqlite.connect(self._path)
            await conn.execute("PRAGMA journal_mode=WAL;")
            await conn.execute(f"PRAGMA busy_timeout={self._busy_timeout_ms};")
            await self._queue.put(conn)
        setup = await self.acquire()
        try:
            await setup.execute(
                "CREATE TABLE IF NOT EXISTS reviews ("
                "id TEXT PRIMARY KEY, repo TEXT, pr_number INTEGER, "
                "decision TEXT, summary TEXT, latency_ms INTEGER, created_at TEXT)"
            )
            await setup.execute(
                "CREATE TABLE IF NOT EXISTS review_comments ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, review_id TEXT, "
                "file TEXT, line INTEGER, severity TEXT, text TEXT)"
            )
            await setup.commit()
        finally:
            await self.release(setup)

    async def acquire(self) -> "aiosqlite.Connection":
        return await self._queue.get()

    async def release(self, conn) -> None:
        await self._queue.put(conn)

    async def close(self) -> None:
        while not self._queue.empty():
            conn = await self._queue.get()
            await conn.close()


# Postgres/asyncpg: shown for contrast only, same pattern as every other
# module in this series — a lazy pool, never opened in this notebook.
#
#   pg_pool: "asyncpg.Pool | None" = None
#   # production equivalent:
#   # pg_pool = await asyncpg.create_pool(dsn=POSTGRES_DSN, min_size=5, max_size=20)

In [ ]:
# prove the pool holds up under real concurrent writers (5 connections, 20 tasks)
import glob

for f in glob.glob("pool_demo.db*"):
    os.remove(f)

_pool_demo = SQLitePool("pool_demo.db", size=5)
await _pool_demo.open()


async def _write(n):
    conn = await _pool_demo.acquire()
    try:
        await conn.execute("INSERT INTO reviews (id, repo, pr_number) VALUES (?, ?, ?)", (f"demo-{n}", "demo/repo", n))
        await conn.commit()
    finally:
        await _pool_demo.release(conn)


t0 = time.perf_counter()
await asyncio.gather(*[_write(n) for n in range(20)])
print(f"20 concurrent writers through a 5-connection pool: {time.perf_counter() - t0:.3f}s")

conn = await _pool_demo.acquire()
cur = await conn.execute("SELECT COUNT(*) FROM reviews")
print("rows written:", (await cur.fetchone())[0])
await _pool_demo.release(conn)
await _pool_demo.close()

for f in glob.glob("pool_demo.db*"):
    os.remove(f)

## LangGraph checkpointer + threads

Separate concern from the pool above: this is the agent's *own* execution
state, not application data. A checkpointer persists the graph's state after
every node, keyed by a `thread_id`. Re-invoking with the same `thread_id`
resumes from the last checkpoint instead of starting fresh — this is what
"threads" means in LangGraph.

DiffSage uses `f"{repo}:{pr_number}"` as the thread id, so re-reviewing a PR
after a new push shares history with the earlier review of the same PR. It's
backed by `aiosqlite` under the hood too, via `langgraph-checkpoint-sqlite`.

In [ ]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import END, StateGraph
import glob


class _DemoState(TypedDict):
    count: int
    history: List[str]


async def _bump(state: _DemoState) -> dict:
    n = state.get("count", 0) + 1
    return {"count": n, "history": state.get("history", []) + [f"pass {n}"]}


for f in glob.glob("checkpoint_demo.db*"):
    os.remove(f)

async with AsyncSqliteSaver.from_conn_string("checkpoint_demo.db") as _demo_cp:
    _b = StateGraph(_DemoState)
    _b.add_node("bump", _bump)
    _b.set_entry_point("bump")
    _b.add_edge("bump", END)
    _demo_graph = _b.compile(checkpointer=_demo_cp)

    thread_a = {"configurable": {"thread_id": "kestrel/billing-service:214"}}
    thread_b = {"configurable": {"thread_id": "kestrel/billing-service:215"}}

    print("thread 214, call 1:", await _demo_graph.ainvoke({"count": 0, "history": []}, config=thread_a))
    print("thread 214, call 2 (same thread, no fresh input):", await _demo_graph.ainvoke({}, config=thread_a))
    print("thread 215, call 1 (different PR, fresh state):  ", await _demo_graph.ainvoke({"count": 0, "history": []}, config=thread_b))

for f in glob.glob("checkpoint_demo.db*"):
    os.remove(f)

## The DiffSage agent

Seven nodes. `check_dependencies` is conditional — it only runs when the
diff touches a manifest file.

`parse_diff` → `retrieve_guidelines` → *(conditional)* `check_dependencies`
→ `lookup_history` → `synthesize_review` *(streamed)* → `persist` → `END`

In [ ]:
MANIFEST_FILES = {"requirements.txt", "package.json", "pyproject.toml", "Cargo.toml", "go.mod"}


class ReviewState(TypedDict):
    repo: str
    pr_number: int
    branch: str
    files_changed: List[str]
    diff: str
    guidelines: List[str]
    advisories: List[str]
    history_note: str
    review_text: str
    decision: str
    comments: List[dict]


async def parse_diff(state: ReviewState) -> dict:
    return {}  # the payload arrives already parsed; this is an explicit pipeline stage


async def retrieve_guidelines(state: ReviewState) -> dict:
    coll = build_style_guide_collection()
    query = " ".join(state["files_changed"]) + " " + state["diff"][:200]
    res = coll.query(query_texts=[query], n_results=2)
    return {"guidelines": res["documents"][0] if res["documents"] else []}


def route_after_guidelines(state: ReviewState) -> str:
    touched = set(state["files_changed"]) & MANIFEST_FILES
    return "check_dependencies" if touched else "lookup_history"


async def check_dependencies(state: ReviewState) -> dict:
    advisories = []
    for line in state["diff"].splitlines():
        line = line.strip().lstrip("+").strip()
        if not line or line.startswith(("diff", "@@", "---", "+++")):
            continue
        if "==" in line:
            note = await check_dependency_advisory(line)
            if note:
                advisories.append(note)
    return {"advisories": advisories}


async def lookup_history(state: ReviewState, pool: "SQLitePool") -> dict:
    conn = await pool.acquire()
    try:
        cur = await conn.execute("SELECT decision, COUNT(*) FROM reviews WHERE repo = ? GROUP BY decision", (state["repo"],))
        rows = await cur.fetchall()
    finally:
        await pool.release(conn)
    if not rows:
        return {"history_note": "no prior reviews for this repo"}
    return {"history_note": "past outcomes: " + ", ".join(f"{c} {d}" for d, c in rows)}


async def synthesize_review(state: ReviewState, model: BaseChatModel) -> dict:
    prompt = (
        f"Review this diff for repo {state['repo']} PR #{state['pr_number']}.\n"
        f"Style guidelines: {state['guidelines']}\n"
        f"Dependency advisories: {state['advisories']}\n"
        f"History: {state['history_note']}\n"
        f"Diff:\n{state['diff']}"
    )
    resp = await model.ainvoke([HumanMessage(content=prompt)])
    decision = "request_changes" if state["advisories"] or state["guidelines"] else "approve"
    comments = []
    if state["advisories"]:
        comments.append({"file": "requirements.txt", "line": 1, "severity": "warning", "text": state["advisories"][0]})
    if state["guidelines"]:
        comments.append({"file": state["files_changed"][0] if state["files_changed"] else "unknown", "line": 1, "severity": "info", "text": state["guidelines"][0]})
    return {"review_text": resp.content, "decision": decision, "comments": comments}


async def persist(state: ReviewState, pool: "SQLitePool", review_id: str) -> dict:
    conn = await pool.acquire()
    try:
        await conn.execute(
            "INSERT INTO reviews (id, repo, pr_number, decision, summary, latency_ms, created_at) VALUES (?, ?, ?, ?, ?, ?, datetime('now'))",
            (review_id, state["repo"], state["pr_number"], state["decision"], state["review_text"][:200], 0),
        )
        for c in state["comments"]:
            await conn.execute(
                "INSERT INTO review_comments (review_id, file, line, severity, text) VALUES (?, ?, ?, ?, ?)",
                (review_id, c["file"], c["line"], c["severity"], c["text"]),
            )
        await conn.commit()
    finally:
        await pool.release(conn)
    return {}

In [ ]:
def build_graph(checkpointer, model: BaseChatModel, pool: "SQLitePool"):
    # Real `async def` closures, not lambdas — see the bug story above.
    async def _lookup_history(state: ReviewState) -> dict:
        return await lookup_history(state, pool)

    async def _synthesize_review(state: ReviewState) -> dict:
        return await synthesize_review(state, model)

    async def _persist(state: ReviewState) -> dict:
        review_id = stable_hash(state["repo"] + str(state["pr_number"]) + str(time.time()))
        return await persist(state, pool, review_id)

    builder = StateGraph(ReviewState)
    builder.add_node("parse_diff", parse_diff)
    builder.add_node("retrieve_guidelines", retrieve_guidelines)
    builder.add_node("check_dependencies", check_dependencies)
    builder.add_node("lookup_history", _lookup_history)
    builder.add_node("synthesize_review", _synthesize_review)
    builder.add_node("persist", _persist)

    builder.set_entry_point("parse_diff")
    builder.add_edge("parse_diff", "retrieve_guidelines")
    builder.add_conditional_edges(
        "retrieve_guidelines", route_after_guidelines,
        {"check_dependencies": "check_dependencies", "lookup_history": "lookup_history"},
    )
    builder.add_edge("check_dependencies", "lookup_history")
    builder.add_edge("lookup_history", "synthesize_review")
    builder.add_edge("synthesize_review", "persist")
    builder.add_edge("persist", END)

    return builder.compile(checkpointer=checkpointer)

## Smoke-testing the agent before wrapping it in FastAPI

Worth confirming the graph works on its own — with a throwaway pool and
checkpointer — before adding an HTTP layer on top of it.

In [ ]:
for f in glob.glob("smoke_*.db*"):
    os.remove(f)

_smoke_pool = SQLitePool("smoke_reviews.db")
await _smoke_pool.open()

async with AsyncSqliteSaver.from_conn_string("smoke_checkpoints.db") as _smoke_cp:
    _smoke_graph = build_graph(_smoke_cp, build_chat_model(), _smoke_pool)
    _result = await _smoke_graph.ainvoke(
        {
            "repo": "kestrel/billing-service", "pr_number": 214, "branch": "main",
            "files_changed": ["billing/webhook.py", "requirements.txt"],
            "diff": "diff --git a/requirements.txt b/requirements.txt\n+requests==2.25.0",
            "guidelines": [], "advisories": [], "history_note": "", "review_text": "", "decision": "", "comments": [],
        },
        config={"configurable": {"thread_id": "kestrel/billing-service:214"}},
    )
    print("decision:", _result["decision"])
    print("comments:", json.dumps(_result["comments"], indent=2))

await _smoke_pool.close()
for f in glob.glob("smoke_*.db*"):
    os.remove(f)

## Wrapping it in FastAPI

`AutoTraceMiddleware` stamps every request with an ID and a latency timer,
adding them as response headers. If `LANGSMITH_TRACING`/`LANGSMITH_API_KEY`
are set it would open a top-level LangSmith run per request; if not, it logs
the same shape locally — optional and gracefully degraded, not a hard
dependency, same as the rest of this middleware's neighbors above.

In [ ]:
from pydantic import BaseModel


class ReviewRequest(BaseModel):
    repo: str
    pr_number: int
    branch: str = "main"
    files_changed: List[str]
    diff: str


class ReviewCommentOut(BaseModel):
    file: str
    line: int
    severity: Literal["info", "warning", "critical"]
    text: str


class ReviewResponse(BaseModel):
    request_id: str
    decision: Literal["approve", "request_changes", "comment"]
    summary: str
    comments: List[ReviewCommentOut]
    latency_ms: int


LOCAL_TRACE_LOG: List[dict] = []


class AutoTraceMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return
        request_id = uuid.uuid4().hex[:8]
        start = time.perf_counter()
        scope["state"] = scope.get("state", {})
        scope["state"]["request_id"] = request_id

        async def send_wrapper(message):
            if message["type"] == "http.response.start":
                latency_ms = int((time.perf_counter() - start) * 1000)
                headers = message.setdefault("headers", [])
                headers.append((b"x-request-id", request_id.encode()))
                headers.append((b"x-latency-ms", str(latency_ms).encode()))
                if not LANGSMITH_LIVE:
                    LOCAL_TRACE_LOG.append({"request_id": request_id, "path": scope["path"], "latency_ms": latency_ms})
                # LANGSMITH_LIVE: a real deployment opens/closes a LangSmith run here, tagged with request_id
            await send(message)

        await self.app(scope, receive, send_wrapper)

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse


@asynccontextmanager
async def lifespan(app: FastAPI):
    pool = SQLitePool(os.getenv("DIFFSAGE_DB", "reviews.db"))
    await pool.open()
    model = build_chat_model()

    async with AsyncSqliteSaver.from_conn_string(os.getenv("DIFFSAGE_CHECKPOINT_DB", "checkpoints.db")) as checkpointer:
        graph = build_graph(checkpointer, model, pool)
        app.state.pool = pool
        app.state.graph = graph
        app.state.mode = {
            "groq": "live" if GROQ_LIVE else "offline",
            "tavily": "live" if TAVILY_LIVE else "offline",
            "langsmith": "live" if LANGSMITH_LIVE else "offline",
        }
        yield

    await pool.close()


app = FastAPI(title="DiffSage", lifespan=lifespan)
app.add_middleware(AutoTraceMiddleware)


@app.get("/health")
async def health():
    return {"status": "ok", "mode": app.state.mode}


def _thread_config(req: ReviewRequest) -> dict:
    return {"configurable": {"thread_id": f"{req.repo}:{req.pr_number}"}}


@app.post("/invoke", response_model=ReviewResponse)
async def invoke(req: ReviewRequest, request: Request):
    t0 = time.perf_counter()
    result = await app.state.graph.ainvoke(
        {"repo": req.repo, "pr_number": req.pr_number, "branch": req.branch, "files_changed": req.files_changed, "diff": req.diff,
         "guidelines": [], "advisories": [], "history_note": "", "review_text": "", "decision": "", "comments": []},
        config=_thread_config(req),
    )
    return ReviewResponse(
        request_id=request.scope.get("state", {}).get("request_id", "unknown"),
        decision=result["decision"] or "comment",
        summary=result["review_text"][:200],
        comments=result["comments"],
        latency_ms=int((time.perf_counter() - t0) * 1000),
    )


@app.post("/stream")
async def stream(req: ReviewRequest, request: Request):
    async def event_gen():
        input_state = {
            "repo": req.repo, "pr_number": req.pr_number, "branch": req.branch,
            "files_changed": req.files_changed, "diff": req.diff,
            "guidelines": [], "advisories": [], "history_note": "", "review_text": "", "decision": "", "comments": [],
        }
        t0 = time.perf_counter()
        node_names = {"parse_diff", "retrieve_guidelines", "check_dependencies", "lookup_history", "synthesize_review", "persist"}
        async for event in app.state.graph.astream_events(input_state, config=_thread_config(req), version="v2"):
            if event["event"] == "on_chain_start" and event["name"] in node_names:
                yield f"data: {json.dumps({'event': 'node_start', 'node': event['name']})}\n\n"
            elif event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield f"data: {json.dumps({'event': 'token', 'content': chunk})}\n\n"
            elif event["event"] == "on_chain_end" and event["name"] == "LangGraph":
                output = event["data"]["output"]
                yield f"data: {json.dumps({'event': 'done', 'decision': output.get('decision'), 'latency_ms': int((time.perf_counter() - t0) * 1000)})}\n\n"

    return StreamingResponse(event_gen(), media_type="text/event-stream")


print("FastAPI app assembled:", [r.path for r in app.routes])

## A standalone `main.py`

Everything above lived in notebook cells for the sake of explaining it piece
by piece. A real deployment needs it as one importable module — this is the
exact file `RUN.md` tells you to start with `uvicorn main:app`.

In [ ]:
%%writefile main.py
"""
DiffSage -- Kestrel Software's AI pull-request review agent.
FastAPI deployment reference: /invoke + /stream, auto-trace middleware,
aiosqlite connection pooling, LangGraph checkpointer + threads.
"""
import asyncio
import hashlib
import json
import os
import time
import uuid
from contextlib import asynccontextmanager
from typing import Any, AsyncIterator, List, Literal, Optional, TypedDict

import aiosqlite
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from dotenv import load_dotenv
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import END, StateGraph
from pydantic import BaseModel

load_dotenv()

GROQ_LIVE = bool(os.getenv("GROQ_API_KEY"))
TAVILY_LIVE = bool(os.getenv("TAVILY_API_KEY"))
LANGSMITH_LIVE = bool(os.getenv("LANGSMITH_API_KEY")) and os.getenv("LANGSMITH_TRACING", "").lower() == "true"


def stable_hash(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:8]


class FakeChatModel(BaseChatModel):
    def _make_reply(self, messages: List[BaseMessage]) -> str:
        last = str(messages[-1].content) if messages else ""
        seed = stable_hash(last)
        return (
            f"[offline review seed={seed}] Found 1 dependency advisory and "
            "1 style guideline note. Recommend request_changes."
        )

    def _generate(self, messages, stop=None, run_manager=None, **kwargs: Any) -> ChatResult:
        text = self._make_reply(messages)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])

    async def _astream(self, messages, stop=None, run_manager=None, **kwargs: Any) -> AsyncIterator[ChatGenerationChunk]:
        text = self._make_reply(messages)
        for word in text.split(" "):
            await asyncio.sleep(0)
            yield ChatGenerationChunk(message=AIMessageChunk(content=word + " "))

    @property
    def _llm_type(self) -> str:
        return "fake-chat-model"


def build_chat_model() -> BaseChatModel:
    if GROQ_LIVE:
        from langchain_groq import ChatGroq
        return ChatGroq(model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"), api_key=os.getenv("GROQ_API_KEY"), streaming=True, temperature=0.2)
    return FakeChatModel()


class DeterministicEmbedder(EmbeddingFunction):
    DIM = 64

    def __init__(self) -> None:
        pass

    def __call__(self, input: Documents) -> Embeddings:
        out = []
        for doc in input:
            vec = [0.0] * self.DIM
            for tok in doc.lower().split():
                h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
                vec[h % self.DIM] += 1.0
            norm = sum(v * v for v in vec) ** 0.5 or 1.0
            out.append([v / norm for v in vec])
        return out


STYLE_GUIDE = [
    "Wrap external HTTP calls to third-party webhooks in the shared retry decorator.",
    "Never catch bare Exception; catch the specific exception type you expect.",
    "Pin dependency versions exactly in requirements.txt, do not use loose ranges.",
    "Database queries in request handlers must go through the connection pool, never a raw sqlite3.connect call.",
    "Log request IDs, never log full request bodies for endpoints that touch customer data.",
]


def build_style_guide_collection():
    client = chromadb.Client()
    coll = client.get_or_create_collection("style_guide", embedding_function=DeterministicEmbedder())
    if coll.count() == 0:
        coll.add(ids=[f"sg{i}" for i in range(len(STYLE_GUIDE))], documents=STYLE_GUIDE)
    return coll


FAKE_ADVISORIES = {
    "requests==2.25.0": "requests==2.25.0 has a known CVE; pin >=2.31.0 instead.",
}


async def check_dependency_advisory(package_line: str) -> Optional[str]:
    if TAVILY_LIVE:
        from tavily import TavilyClient
        client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        result = await asyncio.to_thread(client.search, f"{package_line} security advisory CVE", max_results=1)
        hits = result.get("results", [])
        return hits[0]["content"][:200] if hits else None
    return FAKE_ADVISORIES.get(package_line.strip())


class SQLitePool:
    """N pooled aiosqlite connections. PRAGMA busy_timeout is required, not
    optional -- see the notebook for what happens without it."""

    def __init__(self, path: str, size: int = 5, busy_timeout_ms: int = 5000):
        self._queue: asyncio.Queue = asyncio.Queue(maxsize=size)
        self._path, self._size, self._busy_timeout_ms = path, size, busy_timeout_ms

    async def open(self) -> None:
        for _ in range(self._size):
            conn = await aiosqlite.connect(self._path)
            await conn.execute("PRAGMA journal_mode=WAL;")
            await conn.execute(f"PRAGMA busy_timeout={self._busy_timeout_ms};")
            await self._queue.put(conn)
        setup = await self.acquire()
        try:
            await setup.execute(
                "CREATE TABLE IF NOT EXISTS reviews ("
                "id TEXT PRIMARY KEY, repo TEXT, pr_number INTEGER, "
                "decision TEXT, summary TEXT, latency_ms INTEGER, created_at TEXT)"
            )
            await setup.execute(
                "CREATE TABLE IF NOT EXISTS review_comments ("
                "id INTEGER PRIMARY KEY AUTOINCREMENT, review_id TEXT, "
                "file TEXT, line INTEGER, severity TEXT, text TEXT)"
            )
            await setup.commit()
        finally:
            await self.release(setup)

    async def acquire(self):
        return await self._queue.get()

    async def release(self, conn) -> None:
        await self._queue.put(conn)

    async def close(self) -> None:
        while not self._queue.empty():
            conn = await self._queue.get()
            await conn.close()


# Postgres/asyncpg: shown for contrast only -- a lazy pool, never opened here.
#   pg_pool: "asyncpg.Pool | None" = None
#   # pg_pool = await asyncpg.create_pool(dsn=POSTGRES_DSN, min_size=5, max_size=20)


MANIFEST_FILES = {"requirements.txt", "package.json", "pyproject.toml", "Cargo.toml", "go.mod"}


class ReviewState(TypedDict):
    repo: str
    pr_number: int
    branch: str
    files_changed: List[str]
    diff: str
    guidelines: List[str]
    advisories: List[str]
    history_note: str
    review_text: str
    decision: str
    comments: List[dict]


async def parse_diff(state: ReviewState) -> dict:
    return {}


async def retrieve_guidelines(state: ReviewState) -> dict:
    coll = build_style_guide_collection()
    query = " ".join(state["files_changed"]) + " " + state["diff"][:200]
    res = coll.query(query_texts=[query], n_results=2)
    return {"guidelines": res["documents"][0] if res["documents"] else []}


def route_after_guidelines(state: ReviewState) -> str:
    touched = set(state["files_changed"]) & MANIFEST_FILES
    return "check_dependencies" if touched else "lookup_history"


async def check_dependencies(state: ReviewState) -> dict:
    advisories = []
    for line in state["diff"].splitlines():
        line = line.strip().lstrip("+").strip()
        if not line or line.startswith(("diff", "@@", "---", "+++")):
            continue
        if "==" in line:
            note = await check_dependency_advisory(line)
            if note:
                advisories.append(note)
    return {"advisories": advisories}


async def lookup_history(state: ReviewState, pool: SQLitePool) -> dict:
    conn = await pool.acquire()
    try:
        cur = await conn.execute("SELECT decision, COUNT(*) FROM reviews WHERE repo = ? GROUP BY decision", (state["repo"],))
        rows = await cur.fetchall()
    finally:
        await pool.release(conn)
    if not rows:
        return {"history_note": "no prior reviews for this repo"}
    return {"history_note": "past outcomes: " + ", ".join(f"{c} {d}" for d, c in rows)}


async def synthesize_review(state: ReviewState, model: BaseChatModel) -> dict:
    prompt = (
        f"Review this diff for repo {state['repo']} PR #{state['pr_number']}.\n"
        f"Style guidelines: {state['guidelines']}\n"
        f"Dependency advisories: {state['advisories']}\n"
        f"History: {state['history_note']}\n"
        f"Diff:\n{state['diff']}"
    )
    resp = await model.ainvoke([HumanMessage(content=prompt)])
    decision = "request_changes" if state["advisories"] or state["guidelines"] else "approve"
    comments = []
    if state["advisories"]:
        comments.append({"file": "requirements.txt", "line": 1, "severity": "warning", "text": state["advisories"][0]})
    if state["guidelines"]:
        comments.append({"file": state["files_changed"][0] if state["files_changed"] else "unknown", "line": 1, "severity": "info", "text": state["guidelines"][0]})
    return {"review_text": resp.content, "decision": decision, "comments": comments}


async def persist(state: ReviewState, pool: SQLitePool, review_id: str) -> dict:
    conn = await pool.acquire()
    try:
        await conn.execute(
            "INSERT INTO reviews (id, repo, pr_number, decision, summary, latency_ms, created_at) VALUES (?, ?, ?, ?, ?, ?, datetime('now'))",
            (review_id, state["repo"], state["pr_number"], state["decision"], state["review_text"][:200], 0),
        )
        for c in state["comments"]:
            await conn.execute(
                "INSERT INTO review_comments (review_id, file, line, severity, text) VALUES (?, ?, ?, ?, ?)",
                (review_id, c["file"], c["line"], c["severity"], c["text"]),
            )
        await conn.commit()
    finally:
        await pool.release(conn)
    return {}


def build_graph(checkpointer, model: BaseChatModel, pool: SQLitePool):
    async def _lookup_history(state: ReviewState) -> dict:
        return await lookup_history(state, pool)

    async def _synthesize_review(state: ReviewState) -> dict:
        return await synthesize_review(state, model)

    async def _persist(state: ReviewState) -> dict:
        review_id = stable_hash(state["repo"] + str(state["pr_number"]) + str(time.time()))
        return await persist(state, pool, review_id)

    builder = StateGraph(ReviewState)
    builder.add_node("parse_diff", parse_diff)
    builder.add_node("retrieve_guidelines", retrieve_guidelines)
    builder.add_node("check_dependencies", check_dependencies)
    builder.add_node("lookup_history", _lookup_history)
    builder.add_node("synthesize_review", _synthesize_review)
    builder.add_node("persist", _persist)

    builder.set_entry_point("parse_diff")
    builder.add_edge("parse_diff", "retrieve_guidelines")
    builder.add_conditional_edges(
        "retrieve_guidelines", route_after_guidelines,
        {"check_dependencies": "check_dependencies", "lookup_history": "lookup_history"},
    )
    builder.add_edge("check_dependencies", "lookup_history")
    builder.add_edge("lookup_history", "synthesize_review")
    builder.add_edge("synthesize_review", "persist")
    builder.add_edge("persist", END)

    return builder.compile(checkpointer=checkpointer)


class ReviewRequest(BaseModel):
    repo: str
    pr_number: int
    branch: str = "main"
    files_changed: List[str]
    diff: str


class ReviewCommentOut(BaseModel):
    file: str
    line: int
    severity: Literal["info", "warning", "critical"]
    text: str


class ReviewResponse(BaseModel):
    request_id: str
    decision: Literal["approve", "request_changes", "comment"]
    summary: str
    comments: List[ReviewCommentOut]
    latency_ms: int


LOCAL_TRACE_LOG: List[dict] = []


class AutoTraceMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return
        request_id = uuid.uuid4().hex[:8]
        start = time.perf_counter()
        scope["state"] = scope.get("state", {})
        scope["state"]["request_id"] = request_id

        async def send_wrapper(message):
            if message["type"] == "http.response.start":
                latency_ms = int((time.perf_counter() - start) * 1000)
                headers = message.setdefault("headers", [])
                headers.append((b"x-request-id", request_id.encode()))
                headers.append((b"x-latency-ms", str(latency_ms).encode()))
                if not LANGSMITH_LIVE:
                    LOCAL_TRACE_LOG.append({"request_id": request_id, "path": scope["path"], "latency_ms": latency_ms})
            await send(message)

        await self.app(scope, receive, send_wrapper)


@asynccontextmanager
async def lifespan(app: FastAPI):
    pool = SQLitePool(os.getenv("DIFFSAGE_DB", "reviews.db"))
    await pool.open()
    model = build_chat_model()

    async with AsyncSqliteSaver.from_conn_string(os.getenv("DIFFSAGE_CHECKPOINT_DB", "checkpoints.db")) as checkpointer:
        graph = build_graph(checkpointer, model, pool)
        app.state.pool = pool
        app.state.graph = graph
        app.state.mode = {
            "groq": "live" if GROQ_LIVE else "offline",
            "tavily": "live" if TAVILY_LIVE else "offline",
            "langsmith": "live" if LANGSMITH_LIVE else "offline",
        }
        yield

    await pool.close()


app = FastAPI(title="DiffSage", lifespan=lifespan)
app.add_middleware(AutoTraceMiddleware)


@app.get("/health")
async def health():
    return {"status": "ok", "mode": app.state.mode}


def _thread_config(req: ReviewRequest) -> dict:
    return {"configurable": {"thread_id": f"{req.repo}:{req.pr_number}"}}


@app.post("/invoke", response_model=ReviewResponse)
async def invoke(req: ReviewRequest, request: Request):
    t0 = time.perf_counter()
    result = await app.state.graph.ainvoke(
        {"repo": req.repo, "pr_number": req.pr_number, "branch": req.branch, "files_changed": req.files_changed, "diff": req.diff,
         "guidelines": [], "advisories": [], "history_note": "", "review_text": "", "decision": "", "comments": []},
        config=_thread_config(req),
    )
    return ReviewResponse(
        request_id=request.scope.get("state", {}).get("request_id", "unknown"),
        decision=result["decision"] or "comment",
        summary=result["review_text"][:200],
        comments=result["comments"],
        latency_ms=int((time.perf_counter() - t0) * 1000),
    )


@app.post("/stream")
async def stream(req: ReviewRequest, request: Request):
    async def event_gen():
        input_state = {
            "repo": req.repo, "pr_number": req.pr_number, "branch": req.branch,
            "files_changed": req.files_changed, "diff": req.diff,
            "guidelines": [], "advisories": [], "history_note": "", "review_text": "", "decision": "", "comments": [],
        }
        t0 = time.perf_counter()
        node_names = {"parse_diff", "retrieve_guidelines", "check_dependencies", "lookup_history", "synthesize_review", "persist"}
        async for event in app.state.graph.astream_events(input_state, config=_thread_config(req), version="v2"):
            if event["event"] == "on_chain_start" and event["name"] in node_names:
                yield f"data: {json.dumps({'event': 'node_start', 'node': event['name']})}\n\n"
            elif event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield f"data: {json.dumps({'event': 'token', 'content': chunk})}\n\n"
            elif event["event"] == "on_chain_end" and event["name"] == "LangGraph":
                output = event["data"]["output"]
                yield f"data: {json.dumps({'event': 'done', 'decision': output.get('decision'), 'latency_ms': int((time.perf_counter() - t0) * 1000)})}\n\n"

    return StreamingResponse(event_gen(), media_type="text/event-stream")

## Running it — Uvicorn + real curl

On Windows, from an Anaconda Prompt in this folder, this is just:

```
uvicorn main:app --host 127.0.0.1 --port 8000 --reload
```

Inside the notebook, `uvicorn.Server(...).run()` runs in a background daemon
thread instead — that avoids Windows subprocess/signal quirks entirely — and
the requests below go through real `curl.exe` via `subprocess`, not a
`requests` call standing in for one. Windows 10+ ships `curl.exe` natively,
so these are exactly what you'd type at a prompt.

In [ ]:
import subprocess
import threading
import uvicorn

# a fresh kernel process for main:app -- separate from the app object built
# piecemeal above, so this cell is a true test of the standalone file
_config = uvicorn.Config("main:app", host="127.0.0.1", port=8000, log_level="warning")
_server = uvicorn.Server(_config)
_server_thread = threading.Thread(target=_server.run, daemon=True)
_server_thread.start()

for _ in range(50):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://127.0.0.1:8000/health"], capture_output=True, text=True)
    if r.stdout.strip() == "200":
        break
    time.sleep(0.2)
else:
    raise RuntimeError("server did not become ready")

print(subprocess.run(["curl", "-s", "http://127.0.0.1:8000/health"], capture_output=True, text=True).stdout)

`POST /invoke` — blocking, full JSON back:

In [ ]:
diff_payload = json.dumps({
    "repo": "kestrel/billing-service",
    "pr_number": 214,
    "branch": "feat/stripe-webhook-retry",
    "files_changed": ["billing/webhook.py", "requirements.txt"],
    "diff": "diff --git a/requirements.txt b/requirements.txt\n@@ -9,3 +9,4 @@\n flask==2.0.1\n+requests==2.25.0\n",
})

print("$ curl -X POST http://127.0.0.1:8000/invoke \\")
print("    -H \"Content-Type: application/json\" \\")
print(f"    -d '{diff_payload}'")
print()

r = subprocess.run(
    ["curl", "-s", "-D", "-", "-X", "POST", "http://127.0.0.1:8000/invoke", "-H", "Content-Type: application/json", "-d", diff_payload],
    capture_output=True, text=True,
)
print(r.stdout)

`POST /stream` — same payload, Server-Sent Events back:

In [ ]:
print("$ curl -N -X POST http://127.0.0.1:8000/stream \\")
print("    -H \"Content-Type: application/json\" \\")
print(f"    -d '{diff_payload}'")
print()

r = subprocess.run(
    ["curl", "-s", "-N", "-X", "POST", "http://127.0.0.1:8000/stream", "-H", "Content-Type: application/json", "-d", diff_payload],
    capture_output=True, text=True,
)
print(r.stdout)

Two calls to `/invoke` for the same `repo`/`pr_number` share a thread —
the second one's `lookup_history` node sees the first one's row in
`reviews.db`, and the LangGraph checkpointer carries the run's own state
across the pair under the hood.

In [ ]:
r1 = subprocess.run(["curl", "-s", "-X", "POST", "http://127.0.0.1:8000/invoke", "-H", "Content-Type: application/json",
                      "-d", json.dumps({"repo": "kestrel/checkout-service", "pr_number": 9, "files_changed": ["checkout/cart.py"], "diff": "+ x = 1"})],
                     capture_output=True, text=True)
r2 = subprocess.run(["curl", "-s", "-X", "POST", "http://127.0.0.1:8000/invoke", "-H", "Content-Type: application/json",
                      "-d", json.dumps({"repo": "kestrel/checkout-service", "pr_number": 9, "files_changed": ["checkout/cart.py"], "diff": "+ x = 2"})],
                     capture_output=True, text=True)
print("call 1:", json.loads(r1.stdout)["decision"])
print("call 2:", json.loads(r2.stdout)["decision"])

import sqlite3
conn = sqlite3.connect("reviews.db")
print("rows for kestrel/checkout-service:", conn.execute("SELECT COUNT(*) FROM reviews WHERE repo = ?", ("kestrel/checkout-service",)).fetchone()[0])
conn.close()

In [ ]:
_server.should_exit = True
_server_thread.join(timeout=5)
print("server stopped:", not _server_thread.is_alive())

## What this notebook covered, and what it deliberately left out

Covered, live-tested end to end offline: FastAPI wrapping a LangGraph agent
behind `/invoke` and `/stream`; Uvicorn running locally (background thread
here, `uvicorn main:app --reload` on a real Windows prompt); auto-trace
middleware; an `aiosqlite` connection pool with the `busy_timeout` fix;
LangGraph's own checkpointer and threads for agent-level state.

Left out on purpose, per this module's scope: deep RAG-quality retrieval
(the embedder is intentionally simplistic), a real CI/CD pipeline, and
Bedrock AgentCore. See `RUN.md` for exact commands to run this outside the
notebook, with real Groq/Tavily/LangSmith keys if you have them.